# 05 Hyperbolic Probabilities

Previous notebooks implemented the core HHNN pipeline: hyperbolic states, energy, associative memory, and decision boundaries. This notebook is a simple stochastic extension.

Instead of assigning one deterministic hyperbolic state to an input, we assign probabilities over multiple candidate hyperbolic states.

## From deterministic states to stochastic states

In HHNN, a neuron input is mapped to a single state. Here we consider a probabilistic alternative:

- each candidate hyperbolic state receives a score
- the scores are normalized into probabilities
- this is an exploratory step toward hyperbolic Boltzmann-style modeling


## 1. Quantized hyperbolic state set

We define a small set of candidate hyperbolic states using a practical cosh/sinh parameterization.

In [ ]:
import numpy as np
import pandas as pd

T = 0.5
n_vals = np.arange(-2, 3)

t_vals = n_vals * T
real_part = np.cosh(t_vals)
unipotent_part = np.sinh(t_vals)

states_df = pd.DataFrame({
    "state_index": n_vals,
    "t": t_vals,
    "real_part": real_part,
    "unipotent_part": unipotent_part,
})

states_df

## 2. State compatibility scores

We use a simple exploratory score aligned with hyperbolic interaction ideas:

score(z, I) = exp( - Re(z * I) )

If z = x + h y and I = a + h b, then:

Re(zI) = x·a + y·b

This is a practical demonstration to convert compatibility into probabilities.

In [ ]:
def compatibility_score(state_x, state_y, input_a, input_b):
    # Re(zI) = x*a + y*b
    re = state_x * input_a + state_y * input_b
    return np.exp(-re)


## 3. Example hyperbolic inputs

We define three example inputs: a balanced input, an upper-branch leaning input, and a lower-branch leaning input.

In [ ]:
inputs_df = pd.DataFrame([
    {"input_name": "balanced", "real_part": 1.2, "unipotent_part": 0.0},
    {"input_name": "upper_branch", "real_part": 1.4, "unipotent_part": 0.6},
    {"input_name": "lower_branch", "real_part": 1.4, "unipotent_part": -0.6},
])

inputs_df

## 4. Hyperbolic probabilities over candidate states

For each example input, we compute scores for each candidate state and normalize into probabilities.

In [ ]:
rows = []

for _, row in inputs_df.iterrows():
    a = row["real_part"]
    b = row["unipotent_part"]
    scores = compatibility_score(states_df["real_part"].values, states_df["unipotent_part"].values, a, b)
    probs = scores / np.sum(scores)

    for state_idx, score, prob in zip(states_df["state_index"], scores, probs):
        rows.append({
            "input_name": row["input_name"],
            "state_index": int(state_idx),
            "score": float(score),
            "probability": float(prob),
        })

prob_df = pd.DataFrame(rows)
prob_df

## 5. Probability visualization

We visualize probabilities per input as simple bar charts. Optionally, we show the candidate states on the unit hyperbola and highlight one input.

In [ ]:
import matplotlib.pyplot as plt

for name in inputs_df["input_name"]:
    sub = prob_df[prob_df["input_name"] == name]
    plt.figure(figsize=(5, 3))
    plt.bar(sub["state_index"], sub["probability"], color="C0")
    plt.title(f"State probabilities for {name}")
    plt.xlabel("state_index")
    plt.ylabel("probability")
    plt.tight_layout()
    plt.show()

# Optional geometry view for one input
example = inputs_df.iloc[0]
plt.figure(figsize=(5, 4))

x_curve = np.linspace(1.0, np.max(states_df["real_part"]) * 1.2, 300)
y_curve = np.sqrt(x_curve**2 - 1.0)
plt.plot(x_curve, y_curve, color="gray", linewidth=1.0)
plt.plot(x_curve, -y_curve, color="gray", linewidth=1.0)

plt.scatter(states_df["real_part"], states_df["unipotent_part"], color="C0", label="state points")
plt.scatter(example["real_part"], example["unipotent_part"], color="C1", label="example input")

plt.xlabel("real part")
plt.ylabel("unipotent part")
plt.title("Candidate states and example input")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 6. Deterministic vs probabilistic interpretation

- deterministic HHNN picks a single state
- hyperbolic probabilities assign weight to several states
- nearby states can still receive nonzero probability
- this provides a natural bridge toward stochastic hyperbolic neural models

## Summary

- quantized hyperbolic states were defined
- a compatibility score was computed for each state
- the scores were normalized into probabilities
- this notebook serves as a simple first step toward hyperbolic stochastic modeling